# Width and update interference: 64-neuron teacher

Inspect the first teacher–student experiments. Run this notebook with the repository's `.venv` Python kernel. It can read a sweep while it is still running: rerun the loading cell to refresh.

The saved loss is **half-MSE**. A checkpoint at `step=t` measures gradients at θₜ, then records the actual update θₜ₊₁−θₜ and its loss effects. `step=0` is initialization. The final update is included even if it is not explicitly listed in the configuration.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from neural_competition.diagnostics import alignment_arrays

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "src/neural_competition").is_dir())
EXPERIMENT = Path(os.environ.get("NEURAL_COMPETITION_RUN", ROOT / "runs/teacher64"))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(EXPERIMENT)

## Load results and inspect progress

The protocol and data are shared across the sweep. Initialization and the minibatch sequence are paired by seed across widths and optimizers. Completed and still-running results are included; divergence is recorded explicitly.

In [ ]:
config = json.loads((EXPERIMENT / "config.json").read_text())
environment = json.loads((EXPERIMENT / "environment.json").read_text())
frames, statuses = [], []
for run in sorted(EXPERIMENT.glob("w*_seed*_*")):
    if not (run / "state.json").exists():
        continue
    state = json.loads((run / "state.json").read_text())
    metadata = {"run": run.name, "width": state["width"], "seed": state["seed"],
                "optimizer": state["optimizer"]["label"], "optimizer_name": state["optimizer"]["name"],
                "lr": state["optimizer"]["lr"]}
    statuses.append({**metadata, "status": state["status"], "updates_completed": state["updates_completed"],
                     "final_train_loss": state.get("final_train_loss", np.nan)})
    try:
        frame = pd.read_csv(run / "metrics.csv")
    except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError):
        continue
    for key, value in metadata.items():
        frame[key] = value
    frames.append(frame)
status = pd.DataFrame(statuses)
metrics = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(environment)
display(status)
assert not metrics.empty, "No checkpoint has been saved yet. Wait briefly and rerun this cell."
print(f"{len(metrics)} checkpoints from {metrics['run'].nunique()} runs")

In [ ]:
def plot_metric(column, ylabel, *, log=False):
    labels = [o["label"] for o in config["optimizers"] if o["label"] in set(metrics["optimizer"])]
    fig, axes = plt.subplots(1, len(labels), figsize=(4 * len(labels), 3.2), squeeze=False)
    for ax, label in zip(axes.flat, labels):
        for width, rows in metrics.loc[metrics["optimizer"] == label].groupby("width"):
            summary = rows.groupby("step")[column].agg(["mean", "min", "max"])
            ax.plot(summary.index, summary["mean"], label=f"width {width}")
            ax.fill_between(summary.index, summary["min"], summary["max"], alpha=.12)
        ax.set(title=label, xlabel="Update index", ylabel=ylabel)
        if log:
            ax.set_yscale("log")
        ax.legend(fontsize=8)
    fig.tight_layout()
    return fig

# Shading is the range across seeds, not a confidence interval.
plot_metric("train_loss", "Training half-MSE", log=True)
plot_metric("test_loss", "Test half-MSE", log=True)
plt.show()

## How do the interactions evolve?

Raw inner products depend on gradient magnitudes. Inspect cosine similarities and norms alongside them. The global negative tail can hide negative contributions from individual neurons because other neurons contribute positive scores.

In [ ]:
plot_metric("pair_cosine_negative_fraction", "Fraction of conflicting batch pairs")
plot_metric("pair_dot_q05", "5th percentile of gradient inner products")
plot_metric("gradient_norm_mean", "Mean minibatch gradient norm", log=True)
plot_metric("neuron_negative_mass_mean", "Mean sum of negative neuron contributions")
plt.show()

## Inspect raw distributions at the same checkpoint

Choose a seed, optimizer label, and an **exact** checkpoint step. Runs that have not reached that checkpoint are omitted. Each histogram contains correlated pairs of probe gradients; the number of pairs is not the number of independent experimental repetitions.

In [ ]:
SEED = 0
OPTIMIZER = "sgd_small"
CHECKPOINT_STEP = 0  # Try 100, 700, or 1999 after those checkpoints are available.

snapshots = {}
for width in config["widths"]:
    path = EXPERIMENT / f"w{width:04d}_seed{SEED}_{OPTIMIZER}/checkpoints/step_{CHECKPOINT_STEP:06d}.pt"
    if path.exists():
        snapshots[width] = torch.load(path, map_location="cpu", weights_only=True)
print("Available widths:", list(snapshots))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for width, snapshot in snapshots.items():
    dots, cosines = alignment_arrays(snapshot["probe_gradients"])
    for ax, values in zip(axes, (dots.numpy(), cosines.numpy())):
        values = values[np.isfinite(values)]
        ax.hist(values, bins=40, density=True, histtype="step", label=f"width {width}")
for ax, title in zip(axes, ("Gradient inner product", "Gradient cosine similarity")):
    ax.axvline(0, color="black", lw=.8)
    ax.set(xlabel=title, ylabel="Density", title=f"{OPTIMIZER}, step {CHECKPOINT_STEP}")
    if snapshots:
        ax.legend()
fig.tight_layout()
plt.show()

## Which neurons carry conflicting demands?

For a selected batch pair, qᵣ = gᵦ,ᵣ · g꜀,ᵣ. A negative score means that updating only neuron r for B would harm C to first order. Their sum equals the global inner product. This attributes conflict between batches to shared neurons; it does not by itself show neurons fighting each other.

The plot selects the most negative global pair at this checkpoint. Its neuron scores may include positive and negative contributions.

In [ ]:
if snapshots:
    WIDTH = max(snapshots)
    snapshot = snapshots[WIDTH]
    gradients = snapshot["probe_gradients"]
    flat = gradients.flatten(1)
    gram = flat @ flat.T
    pairs = torch.triu_indices(len(flat), len(flat), offset=1)
    pair_scores = gram[pairs[0], pairs[1]]
    k = int(pair_scores.argmin())
    i, j = int(pairs[0, k]), int(pairs[1, k])
    neuron_scores = (gradients[i] * gradients[j]).sum(dim=1)
    torch.testing.assert_close(neuron_scores.sum(), gram[i, j], rtol=2e-4, atol=1e-6)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    image = axes[0].imshow(gram.numpy(), cmap="coolwarm")
    fig.colorbar(image, ax=axes[0], label="Gradient inner product")
    axes[0].set(title=f"Width {WIDTH}: batch interactions", xlabel="Probe batch", ylabel="Probe batch")
    axes[1].bar(np.arange(WIDTH), neuron_scores.numpy(), color=np.where(neuron_scores.numpy() < 0, "tab:red", "tab:blue"))
    axes[1].axhline(0, color="black", lw=.8)
    axes[1].set(title=f"Neuron attribution for batches {i}, {j}", xlabel="Neuron", ylabel="Contribution to inner product")
    fig.tight_layout()
    plt.show()
else:
    print("Choose an available checkpoint above.")

## Does the actual update damage other batches?

The horizontal coordinate is g꜀ · Δθ, the first-order prediction of the loss change. Positive means predicted harm. The vertical coordinate is the observed L꜀(θ+Δθ)−L꜀(θ). This uses the actual SGD, momentum, or Adam update.

The difference includes second-order and higher-order terms and possible ReLU activation-boundary crossings. It should not be interpreted as a pure second-order term.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
effects = []
for width, snapshot in snapshots.items():
    predicted = snapshot["probe_gradients"].flatten(1) @ snapshot["actual_step"].flatten()
    observed = snapshot["probe_losses_after"] - snapshot["probe_losses_before"]
    ax.scatter(predicted.numpy(), observed.numpy(), s=14, alpha=.6, label=f"width {width}")
    effects.append({"width": width, "predicted_harm_fraction": float((predicted > 0).float().mean()),
                    "observed_harm_fraction": float((observed > 0).float().mean()),
                    "mean_loss_change": float(observed.mean()),
                    "mean_remainder": float((observed - predicted).mean())})
if snapshots:
    low, high = ax.get_xlim()
    ax.plot([low, high], [low, high], "k--", lw=1, label="First-order prediction")
    ax.legend(fontsize=8)
ax.axhline(0, color="gray", lw=.5)
ax.axvline(0, color="gray", lw=.5)
ax.set(xlabel="Predicted loss change", ylabel="Observed loss change")
fig.tight_layout()
plt.show()
display(pd.DataFrame(effects))
plot_metric("probe_loss_increase_fraction", "Fraction of probe batches harmed by actual step")
plt.show()

## Curvature and optimizer memory

Batch sharpness is the **mean of Rayleigh quotients** gᵦᵀHᵦgᵦ / ‖gᵦ‖², using each batch's own Hessian and gradient. It is estimated on a small subset of probe batches on CPU in float64.

The plain-SGD reference level 2/η is shown only for plain SGD. It is not a universal threshold for momentum or Adam, and a noisy plateau alone does not certify EoSS. Negative curvature values are possible in this nonconvex network.

In [ ]:
fig = plot_metric("batch_sharpness", "Batch sharpness")
labels = [o["label"] for o in config["optimizers"] if o["label"] in set(metrics["optimizer"])]
for ax, label in zip(fig.axes, labels):
    spec = next(o for o in config["optimizers"] if o["label"] == label)
    if spec["name"] == "sgd":
        ax.axhline(2 / spec["lr"], color="black", linestyle="--", lw=1, label="2 / learning rate")
        ax.legend(fontsize=8)
plt.show()
plot_metric("step_previous_cosine", "Cosine of consecutive parameter steps")
plt.show()
# Raw buffers remain available for further analysis:
if snapshots:
    print({key: tuple(value.shape) for key, value in next(iter(snapshots.values()))["optimizer_before"].items()})

## Compare widths at similar loss

Choose a target loss and inspect the nearest measured checkpoint in each run. The table includes the mismatch explicitly. A nearest checkpoint is not automatically a sufficiently close match; tighten checkpoint spacing if necessary. Consider useful progress and step magnitude alongside alignment.

In [ ]:
TARGET_LOSS = 0.1
selected = metrics.assign(loss_gap=(metrics["train_loss"] - TARGET_LOSS).abs())
selected = selected.loc[selected.groupby("run")["loss_gap"].idxmin()]
columns = ["width", "seed", "optimizer", "step", "train_loss", "loss_gap", "pair_cosine_negative_fraction",
           "pair_dot_q05", "probe_loss_increase_fraction", "step_norm", "batch_sharpness"]
display(selected[columns].sort_values(["optimizer", "width", "seed"]))

## Interpretation checklist

- Does width improve training speed or final error in this setting?
- Does it change the negative tail of alignment, beyond a change in gradient norms?
- Is the change present at initialization, learned during training, or mostly explained by different losses?
- Which neurons carry the conflicting demands, and does their attribution change?
- Does actual cross-batch damage follow the first-order prediction? Does optimizer memory change this?
- Is any change associated with curvature? A correlation does not establish a causal mechanism.

The initial outgoing weights are zero, making predictions and initial loss identical across widths. Hidden-weight gradients are zero at the first update and become active afterward. This is a controlled initialization choice, not a universal model of neural training. The width scaling is 1/√m, so conclusions should be checked against other parameterizations before being generalized.

The first sweep compares SGD and momentum. Add Adam through the JSON configuration once the pilot is understood; optimizer-state tensors and actual-step diagnostics already support it.